# Fine-Tuning Healthcare Language Models: A Practical Guide

This notebook demonstrates how to:
1. Fine-tune a **healthcare-specific** pre-trained model
2. Use **parameter-efficient fine-tuning (LoRA)** to reduce compute requirements
3. Benchmark against a baseline and evaluate with standard metrics

## Models We'll Use

| Model | Parameters | Domain | Use Case |
|-------|------------|--------|----------|
| **PubMedBERT** | 110M | Biomedical | Classification (primary example) |
| **BioGPT** | 347M | Biomedical | Text generation |
| **BioMistral-7B** | 7B | Healthcare | Advanced (with QLoRA) |

We'll start with PubMedBERT for the main walkthrough, then show how to scale to larger models.

## Background: Why Healthcare-Specific Models?

General-purpose models (BERT, GPT) are trained on web text, Wikipedia, and books. They may not understand:
- Medical terminology ("dyspnea", "hepatomegaly")
- Clinical abbreviations ("bid", "prn", "SOB")
- Biomedical relationships (drug-gene interactions)

**Healthcare-specific models** are pre-trained on:
- PubMed abstracts (30M+ articles)
- Clinical notes (MIMIC-III)
- Biomedical literature

This domain-specific pre-training provides a better starting point for fine-tuning.

---
## Setup and Installation

In [ ]:
# Install required packages (run once)
# !pip install transformers datasets torch scikit-learn pandas matplotlib seaborn
# !pip install peft accelerate bitsandbytes  # For LoRA and quantization

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, roc_auc_score,
    ConfusionMatrixDisplay, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, pipeline,
    BitsAndBytesConfig
)
from datasets import Dataset

# Check hardware
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## Step 1: Prepare Biomedical Dataset

### Task: Classify Medical Abstracts by Study Type

We'll classify PubMed-style abstracts into three categories:
- **Treatment**: Therapeutic intervention studies
- **Diagnosis**: Diagnostic accuracy studies  
- **Prognosis**: Outcome prediction studies

This is a real task used in systematic review automation.

In [ ]:
# Medical abstracts dataset (synthetic for demonstration)
# In practice, use real data from PubMed via Entrez API

treatment_abstracts = [
    "BACKGROUND: Type 2 diabetes mellitus requires effective glycemic control. METHODS: This randomized controlled trial evaluated metformin versus placebo in 500 patients over 12 months. RESULTS: HbA1c decreased by 1.2% in the treatment group (p<0.001). CONCLUSION: Metformin significantly improves glycemic control.",
    "OBJECTIVE: To assess the efficacy of cognitive behavioral therapy (CBT) for major depression. DESIGN: Double-blind RCT with 200 participants. INTERVENTION: 12 weeks of CBT versus standard care. OUTCOMES: Depression scores improved significantly in CBT group (Cohen's d=0.8).",
    "AIM: Evaluate pembrolizumab in advanced non-small cell lung cancer. METHODS: Phase III trial comparing pembrolizumab with docetaxel. RESULTS: Median overall survival was 12.7 vs 8.5 months (HR=0.61). CONCLUSION: Pembrolizumab improves survival in PD-L1 positive NSCLC.",
    "PURPOSE: Compare laparoscopic versus open appendectomy. PATIENTS: 300 patients with acute appendicitis randomized 1:1. RESULTS: Operative time was similar; hospital stay shorter with laparoscopy (1.2 vs 2.4 days, p<0.01). CONCLUSION: Laparoscopic approach reduces recovery time.",
    "BACKGROUND: Heart failure with preserved ejection fraction lacks effective treatments. METHODS: EMPEROR-Preserved trial of empagliflozin in 5988 patients. RESULTS: Primary endpoint reduced by 21% (HR 0.79, 95% CI 0.69-0.90). CONCLUSION: SGLT2 inhibitors benefit HFpEF patients.",
    "OBJECTIVE: Assess rituximab for rheumatoid arthritis. DESIGN: Multicenter RCT, 24-week follow-up. INTERVENTION: Rituximab 1000mg IV versus placebo. RESULTS: ACR50 response achieved in 43% vs 13% (p<0.0001). CONCLUSION: Rituximab is effective for refractory RA.",
    "AIM: Evaluate bariatric surgery versus lifestyle intervention for severe obesity. METHODS: 150 patients randomized, 5-year follow-up. RESULTS: Weight loss 25% vs 8%; diabetes remission 75% vs 15%. CONCLUSION: Surgery superior for sustained weight loss.",
    "PURPOSE: Test nivolumab plus ipilimumab in metastatic melanoma. DESIGN: CheckMate 067 trial, 945 patients. RESULTS: 5-year survival 52% combination vs 26% ipilimumab alone. CONCLUSION: Combination immunotherapy improves long-term melanoma survival.",
    "BACKGROUND: Acute migraine requires fast-acting treatment. METHODS: RCT of ubrogepant versus placebo in 1672 patients. RESULTS: Pain freedom at 2h: 19.2% vs 11.8% (p<0.001). CONCLUSION: Ubrogepant effective for acute migraine.",
    "OBJECTIVE: Compare TAVR versus surgical aortic valve replacement. METHODS: 1000 low-risk patients randomized. RESULTS: Death or disabling stroke at 2 years: 5.3% vs 6.7% (non-inferior). CONCLUSION: TAVR is a valid alternative to surgery.",
    "AIM: Evaluate dupilumab for moderate-to-severe atopic dermatitis. DESIGN: Phase III RCT, 16 weeks. RESULTS: EASI-75 achieved in 44% vs 12% placebo (p<0.001). CONCLUSION: Dupilumab significantly improves atopic dermatitis.",
    "PURPOSE: Assess osimertinib in EGFR-mutated lung cancer. METHODS: FLAURA trial, 556 patients. RESULTS: Median PFS 18.9 vs 10.2 months with standard EGFR-TKI. CONCLUSION: Osimertinib should be first-line for EGFR+ NSCLC.",
]

diagnosis_abstracts = [
    "OBJECTIVE: Evaluate liquid biopsy for early lung cancer detection. METHODS: Prospective study of 1000 high-risk individuals; ctDNA analysis compared to CT screening. RESULTS: Sensitivity 89%, specificity 95%, AUC 0.94. CONCLUSION: Liquid biopsy shows promise for lung cancer screening.",
    "AIM: Assess diagnostic accuracy of MRI for prostate cancer. DESIGN: Multi-center study, 500 patients with elevated PSA. REFERENCE: Systematic biopsy. RESULTS: PI-RADS ≥3 sensitivity 93%, specificity 41%, NPV 89%. CONCLUSION: MRI can reduce unnecessary biopsies.",
    "PURPOSE: Validate a machine learning algorithm for diabetic retinopathy detection. METHODS: 10,000 fundus images from diabetic patients. RESULTS: Algorithm achieved AUC 0.97, sensitivity 95%, specificity 91%. CONCLUSION: AI screening is highly accurate for DR detection.",
    "BACKGROUND: Early Alzheimer's diagnosis remains challenging. METHODS: Evaluated plasma p-tau217 in 800 participants. RESULTS: AUC 0.89 for distinguishing AD from controls; correlated with amyloid PET (r=0.72). CONCLUSION: Plasma p-tau217 is a promising blood biomarker.",
    "OBJECTIVE: Compare rapid antigen test to PCR for COVID-19. DESIGN: Diagnostic accuracy study, 2000 symptomatic patients. RESULTS: Sensitivity 82% (95% CI 78-86%), specificity 99.5%. Sensitivity higher with Ct<25 (95%). CONCLUSION: Antigen tests reliable with high viral load.",
    "AIM: Assess troponin I for diagnosing myocardial infarction. METHODS: Prospective cohort, 1500 chest pain patients in ED. RESULTS: High-sensitivity assay at 99th percentile: sensitivity 95%, NPV 99.5%. CONCLUSION: hs-TnI effectively rules out MI.",
    "PURPOSE: Evaluate FDG-PET for lymphoma staging. PATIENTS: 300 newly diagnosed lymphoma patients. REFERENCE: CT plus biopsy. RESULTS: PET upstaged 15%, downstaged 8%; changed management in 23%. CONCLUSION: PET should be standard for lymphoma staging.",
    "BACKGROUND: Celiac disease underdiagnosis is common. METHODS: Compared tissue transglutaminase IgA with duodenal biopsy in 600 patients. RESULTS: TTG-IgA >10x ULN: sensitivity 98%, PPV 99% for villous atrophy. CONCLUSION: Very high TTG may obviate biopsy.",
    "OBJECTIVE: Assess dermoscopy for melanoma detection. DESIGN: Meta-analysis of 15 studies, 8000 lesions. RESULTS: Pooled sensitivity 89%, specificity 79% for dermoscopy vs 71%/81% for naked eye. CONCLUSION: Dermoscopy improves melanoma detection.",
    "AIM: Validate SOFA score for sepsis diagnosis. METHODS: Retrospective cohort, 5000 ICU admissions. RESULTS: SOFA ≥2 for sepsis: AUC 0.74, sensitivity 82%, specificity 63%. CONCLUSION: SOFA helps identify sepsis but has moderate specificity.",
    "PURPOSE: Evaluate circulating tumor cells for breast cancer detection. METHODS: 500 patients with suspicious mammograms. RESULTS: CTC count ≥5: sensitivity 67%, specificity 96% for malignancy. CONCLUSION: CTC analysis may complement imaging.",
    "BACKGROUND: Non-invasive fibrosis assessment needed in NAFLD. METHODS: Compared FIB-4 score with liver biopsy in 800 patients. RESULTS: FIB-4 <1.3 ruled out advanced fibrosis (NPV 95%); >2.67 ruled in (PPV 80%). CONCLUSION: FIB-4 useful for fibrosis triage.",
]

prognosis_abstracts = [
    "OBJECTIVE: Identify prognostic factors in glioblastoma. METHODS: Retrospective analysis of 500 patients; molecular profiling and survival analysis. RESULTS: MGMT methylation associated with improved survival (18 vs 12 months, HR 0.6). IDH mutation favorable. CONCLUSION: Molecular markers predict GBM outcomes.",
    "AIM: Develop cardiovascular risk prediction model for diabetics. DESIGN: Prospective cohort, 10,000 patients, 10-year follow-up. RESULTS: Model including HbA1c, duration, nephropathy achieved C-statistic 0.78. CONCLUSION: Diabetes-specific model improves CV risk prediction.",
    "PURPOSE: Assess tumor-infiltrating lymphocytes as prognostic marker. METHODS: Analysis of 2000 breast cancer specimens. RESULTS: High TILs associated with improved DFS (HR 0.55) and OS (HR 0.60) in triple-negative subtype. CONCLUSION: TILs predict outcomes in TNBC.",
    "BACKGROUND: Heart failure readmission prediction is crucial. METHODS: ML model developed on 50,000 HF admissions. FEATURES: Labs, vitals, comorbidities. RESULTS: AUC 0.72 for 30-day readmission; top predictors: BNP, renal function. CONCLUSION: ML can identify high-risk HF patients.",
    "OBJECTIVE: Identify predictors of COVID-19 mortality. DESIGN: Multi-center cohort, 5000 hospitalized patients. RESULTS: Age, D-dimer, lymphopenia, and IL-6 independently predicted death. Risk score AUC 0.82. CONCLUSION: Simple markers stratify COVID-19 mortality risk.",
    "AIM: Evaluate ctDNA for predicting colorectal cancer recurrence. METHODS: Longitudinal ctDNA monitoring in 200 stage II-III patients post-surgery. RESULTS: ctDNA positivity predicted recurrence (HR 7.2, p<0.001); lead time 8 months. CONCLUSION: ctDNA detects recurrence before imaging.",
    "PURPOSE: Develop prognostic model for acute kidney injury. PATIENTS: 20,000 hospitalized patients. RESULTS: Model with creatinine trajectory, urine output, and comorbidities: AUC 0.85 for severe AKI. CONCLUSION: Dynamic model improves AKI prediction.",
    "BACKGROUND: Stroke outcome prediction needed for resource planning. METHODS: NIHSS, age, and imaging features in 3000 patients. RESULTS: Model predicted poor outcome (mRS>3) with AUC 0.81. NIHSS most important predictor. CONCLUSION: Clinical-imaging model predicts stroke disability.",
    "OBJECTIVE: Identify gene signature for breast cancer recurrence. METHODS: RNA sequencing of 1000 tumors; developed 21-gene signature. RESULTS: High-risk score: 10-year recurrence 30% vs 5% low-risk (HR 4.5). CONCLUSION: Gene signature guides adjuvant therapy decisions.",
    "AIM: Assess frailty as predictor of surgical outcomes. DESIGN: Prospective study, 2000 elderly surgery patients. RESULTS: Frailty index predicted mortality (OR 3.2), complications (OR 2.5), and prolonged stay. CONCLUSION: Frailty assessment essential in surgical planning.",
    "PURPOSE: Evaluate PD-L1 as prognostic marker in NSCLC. METHODS: Immunohistochemistry in 800 resected tumors. RESULTS: High PD-L1 associated with better immunotherapy response but worse prognosis without treatment. CONCLUSION: PD-L1 is predictive, not purely prognostic.",
    "BACKGROUND: Predict progression in mild cognitive impairment. METHODS: Combined MRI, CSF biomarkers, and cognition in 500 MCI patients. RESULTS: Model predicted AD conversion at 3 years (AUC 0.88). Hippocampal volume and p-tau strongest predictors. CONCLUSION: Multimodal assessment predicts MCI progression.",
]

# Create dataset
data = pd.DataFrame({
    'text': treatment_abstracts + diagnosis_abstracts + prognosis_abstracts,
    'label': [0]*len(treatment_abstracts) + [1]*len(diagnosis_abstracts) + [2]*len(prognosis_abstracts)
})

label_names = ['Treatment', 'Diagnosis', 'Prognosis']
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

# Shuffle
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset: {len(data)} medical abstracts")
print(f"\nLabel distribution:")
for i, name in enumerate(label_names):
    count = (data['label'] == i).sum()
    print(f"  {name}: {count} ({count/len(data)*100:.1f}%)")

In [ ]:
# Train/validation/test split
train_df, temp_df = train_test_split(data, test_size=0.3, random_state=42, stratify=data['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"Training:   {len(train_df)} examples")
print(f"Validation: {len(val_df)} examples")
print(f"Test:       {len(test_df)} examples")

# Convert to HuggingFace datasets
train_dataset = Dataset.from_pandas(train_df[['text', 'label']].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[['text', 'label']].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[['text', 'label']].reset_index(drop=True))

---
## Step 2: Baseline - General Purpose Model (Zero-Shot)

First, let's see how a **general-purpose model** performs without any fine-tuning. This establishes our baseline.

In [ ]:
print("Loading zero-shot classifier (BART-MNLI)...")
zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if device == "cuda" else -1
)

# Zero-shot labels (natural language descriptions)
zs_labels = [
    "therapeutic treatment intervention study",
    "diagnostic test accuracy study", 
    "prognostic outcome prediction study"
]

print("Running baseline predictions...")
baseline_preds = []
for text in test_df['text'].values:
    result = zero_shot(text, zs_labels)
    pred = zs_labels.index(result['labels'][0])
    baseline_preds.append(pred)

baseline_preds = np.array(baseline_preds)
print("Done!")

In [ ]:
# Baseline metrics
baseline_acc = accuracy_score(test_df['label'], baseline_preds)
baseline_prec, baseline_rec, baseline_f1, _ = precision_recall_fscore_support(
    test_df['label'], baseline_preds, average='weighted'
)

print("=" * 60)
print("BASELINE: General Model (Zero-Shot, No Fine-Tuning)")
print("=" * 60)
print(f"Accuracy:  {baseline_acc:.3f}")
print(f"Precision: {baseline_prec:.3f}")
print(f"Recall:    {baseline_rec:.3f}")
print(f"F1 Score:  {baseline_f1:.3f}")
print("\n" + classification_report(test_df['label'], baseline_preds, target_names=label_names))

---
## Step 3: Fine-Tune PubMedBERT (Healthcare-Specific Model)

### About PubMedBERT

**PubMedBERT** ([Microsoft Research](https://huggingface.co/microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext)) is:
- Pre-trained **from scratch** on PubMed abstracts and full-text articles
- NOT initialized from general BERT (domain-specific from the start)
- 110M parameters (same size as BERT-base)
- State-of-the-art on many biomedical NLP benchmarks

This is more suitable than general BERT for medical text.

In [ ]:
# Load PubMedBERT
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

# Move to GPU if available
model = model.to(device)

print(f"Model loaded: {model.num_parameters():,} parameters")
print(f"Device: {next(model.parameters()).device}")

In [ ]:
# Tokenize datasets
def tokenize_fn(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512  # PubMedBERT supports 512 tokens
    )

print("Tokenizing...")
train_tok = train_dataset.map(tokenize_fn, batched=True)
val_tok = val_dataset.map(tokenize_fn, batched=True)
test_tok = test_dataset.map(tokenize_fn, batched=True)

# Format for PyTorch
for ds in [train_tok, val_tok, test_tok]:
    ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("Done!")

In [ ]:
# Metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

# Training arguments
training_args = TrainingArguments(
    output_dir='./pubmedbert-medical-classifier',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='none',
    fp16=torch.cuda.is_available(),  # Mixed precision if GPU available
)

print("Training Configuration:")
print(f"  Epochs:        {training_args.num_train_epochs}")
print(f"  Batch size:    {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Mixed precision (fp16): {training_args.fp16}")

In [ ]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)

print("="*60)
print("FINE-TUNING PubMedBERT")
print("="*60)

# Train
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"Time: {train_result.metrics['train_runtime']:.1f} seconds")
print(f"Samples/second: {train_result.metrics['train_samples_per_second']:.1f}")

---
## Step 4: Evaluate Fine-Tuned Model

In [ ]:
# Get predictions
print("Evaluating on test set...")
predictions = trainer.predict(test_tok)

finetuned_preds = np.argmax(predictions.predictions, axis=1)
finetuned_probs = torch.softmax(torch.tensor(predictions.predictions), dim=1).numpy()

# Metrics
ft_acc = accuracy_score(test_df['label'], finetuned_preds)
ft_prec, ft_rec, ft_f1, _ = precision_recall_fscore_support(
    test_df['label'], finetuned_preds, average='weighted'
)

print("\n" + "="*60)
print("FINE-TUNED PubMedBERT PERFORMANCE")
print("="*60)
print(f"Accuracy:  {ft_acc:.3f}")
print(f"Precision: {ft_prec:.3f}")
print(f"Recall:    {ft_rec:.3f}")
print(f"F1 Score:  {ft_f1:.3f}")
print("\n" + classification_report(test_df['label'], finetuned_preds, target_names=label_names))

---
## Step 5: Comprehensive Model Comparison

In [ ]:
# Comparison table
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Baseline (Zero-Shot)': [baseline_acc, baseline_prec, baseline_rec, baseline_f1],
    'PubMedBERT (Fine-Tuned)': [ft_acc, ft_prec, ft_rec, ft_f1]
})

comparison['Absolute Gain'] = comparison['PubMedBERT (Fine-Tuned)'] - comparison['Baseline (Zero-Shot)']
comparison['% Improvement'] = (comparison['Absolute Gain'] / comparison['Baseline (Zero-Shot)'] * 100)

print("="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)
print(comparison.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print("\n")
print(f"Key Finding: Fine-tuning improved F1 by {comparison.loc[comparison['Metric']=='F1 Score', '% Improvement'].values[0]:.1f}%")

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Bar chart comparison
ax = axes[0, 0]
x = np.arange(4)
width = 0.35
bars1 = ax.bar(x - width/2, comparison['Baseline (Zero-Shot)'], width, label='Baseline', color='steelblue', alpha=0.7)
bars2 = ax.bar(x + width/2, comparison['PubMedBERT (Fine-Tuned)'], width, label='Fine-Tuned', color='coral', alpha=0.7)
ax.set_ylabel('Score')
ax.set_title('Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1'])
ax.legend()
ax.set_ylim(0, 1.1)
ax.axhline(y=0.33, color='gray', linestyle='--', alpha=0.5)
ax.text(3.5, 0.35, 'Random', fontsize=9, color='gray')

# 2. Confusion Matrix - Baseline
ax = axes[0, 1]
cm_base = confusion_matrix(test_df['label'], baseline_preds)
sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=label_names, yticklabels=label_names)
ax.set_title('Baseline Confusion Matrix')
ax.set_ylabel('True')
ax.set_xlabel('Predicted')

# 3. Confusion Matrix - Fine-tuned
ax = axes[1, 0]
cm_ft = confusion_matrix(test_df['label'], finetuned_preds)
sns.heatmap(cm_ft, annot=True, fmt='d', cmap='Oranges', ax=ax,
            xticklabels=label_names, yticklabels=label_names)
ax.set_title('Fine-Tuned Confusion Matrix')
ax.set_ylabel('True')
ax.set_xlabel('Predicted')

# 4. ROC Curves
ax = axes[1, 1]
y_test_bin = label_binarize(test_df['label'], classes=[0, 1, 2])
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for i, (name, color) in enumerate(zip(label_names, colors)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], finetuned_probs[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={roc_auc:.2f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves (Fine-Tuned Model)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate overall AUC
overall_auc = roc_auc_score(y_test_bin, finetuned_probs, average='weighted', multi_class='ovr')
print(f"\nWeighted ROC-AUC: {overall_auc:.3f}")

---
## Step 6: (Advanced) Fine-Tuning Larger Models with LoRA

For larger models (1B+ parameters), **full fine-tuning** is often impractical. 

**LoRA (Low-Rank Adaptation)** solves this by:
1. Freezing the original model weights
2. Adding small trainable "adapter" matrices
3. Training only ~1% of parameters

This enables fine-tuning 7B models on consumer GPUs (16GB VRAM).

### Example: BioMistral-7B with QLoRA

Below is the setup for fine-tuning a 7B healthcare model. **Requires ~16GB GPU RAM.**

In [ ]:
# This cell shows the SETUP for LoRA fine-tuning of larger models
# Uncomment and run if you have sufficient GPU memory (16GB+)

LARGE_MODEL_EXAMPLE = False  # Set to True if you have a capable GPU

if LARGE_MODEL_EXAMPLE:
    from peft import LoraConfig, get_peft_model, TaskType
    
    # Model options for healthcare:
    # - "BioMistral/BioMistral-7B" (7B, medical)
    # - "epfl-llm/meditron-7b" (7B, medical)
    # - "microsoft/biogpt" (347M, biomedical generation)
    
    LARGE_MODEL = "microsoft/biogpt"  # Smaller option for demo
    
    # Quantization config (reduces memory by ~4x)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    
    # LoRA config
    lora_config = LoraConfig(
        r=16,                # Rank of adaptation matrices
        lora_alpha=32,       # Scaling factor
        target_modules=["q_proj", "v_proj"],  # Which layers to adapt
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_CLS
    )
    
    print("LoRA Configuration:")
    print(f"  Rank (r): {lora_config.r}")
    print(f"  Alpha: {lora_config.lora_alpha}")
    print(f"  Target modules: {lora_config.target_modules}")
    print(f"\nThis reduces trainable parameters by ~99%!")
else:
    print("LoRA example skipped (set LARGE_MODEL_EXAMPLE=True to run)")
    print("\nLoRA enables fine-tuning 7B models on 16GB GPUs by:")
    print("  - Freezing original weights")
    print("  - Adding small trainable adapters (~1% of params)")
    print("  - Using 4-bit quantization (QLoRA)")

---
## Step 7: Using the Fine-Tuned Model

In [ ]:
# Create inference pipeline
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if device == "cuda" else -1
)

# Test on new abstracts
test_abstracts = [
    "This phase II trial evaluated the safety and efficacy of a novel JAK inhibitor in patients with moderate-to-severe rheumatoid arthritis who had inadequate response to methotrexate.",
    "We assessed the diagnostic performance of a deep learning algorithm for detecting acute intracranial hemorrhage on non-contrast CT scans in emergency department patients.",
    "This cohort study identified clinical and biomarker predictors of rapid disease progression in patients with amyotrophic lateral sclerosis over a 24-month follow-up period.",
]

print("\n" + "="*70)
print("INFERENCE ON NEW ABSTRACTS")
print("="*70)

for i, abstract in enumerate(test_abstracts, 1):
    result = classifier(abstract)[0]
    print(f"\n[Abstract {i}]")
    print(f"Text: {abstract[:100]}...")
    print(f"Prediction: {result['label']}")
    print(f"Confidence: {result['score']:.1%}")

---
## Summary: Key Takeaways

### Performance Metrics Explained

| Metric | Formula | Use When |
|--------|---------|----------|
| **Accuracy** | Correct / Total | Classes are balanced |
| **Precision** | TP / (TP + FP) | False positives are costly |
| **Recall** | TP / (TP + FN) | False negatives are costly |
| **F1 Score** | 2 × (P × R) / (P + R) | Need balance of P and R |
| **ROC-AUC** | Area under ROC curve | Comparing models, threshold-free |

### What We Demonstrated

1. **Domain-specific models** (PubMedBERT) outperform general models on medical tasks
2. **Fine-tuning** dramatically improves performance over zero-shot
3. **Small datasets** (36 examples!) can still yield meaningful improvements
4. **LoRA** enables fine-tuning larger models on limited hardware

### Healthcare Model Options

| Model | Size | Specialty | Hugging Face ID |
|-------|------|-----------|----------------|
| PubMedBERT | 110M | Biomedical literature | `microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext` |
| BioBERT | 110M | Biomedical NER/QA | `dmis-lab/biobert-base-cased-v1.2` |
| ClinicalBERT | 110M | Clinical notes | `emilyalsentzer/Bio_ClinicalBERT` |
| BioGPT | 347M | Biomedical generation | `microsoft/biogpt` |
| BioMistral | 7B | Medical QA | `BioMistral/BioMistral-7B` |
| Meditron | 7B/70B | Medical reasoning | `epfl-llm/meditron-7b` |

---
## In-Class Exercise

### Part 1: Experiment with Hyperparameters
1. Change `num_train_epochs` to 3 and 10. How does F1 change?
2. Try `learning_rate` values of 1e-5 and 5e-5
3. Record results in a table and identify the best configuration

### Part 2: Try a Different Healthcare Model
1. Replace PubMedBERT with ClinicalBERT: `emilyalsentzer/Bio_ClinicalBERT`
2. Compare performance - which model is better for this task?
3. Why might one model outperform the other?

### Part 3: Error Analysis
1. Look at the confusion matrix - which categories are confused most often?
2. Examine misclassified examples - what makes them difficult?
3. How could you improve the model or dataset?

### Part 4: Discussion Questions
1. With only 36 training examples, are we at risk of overfitting? How would you check?
2. What additional data would improve this classifier?
3. How would you deploy this model in a real literature screening workflow?

In [ ]:
# Space for your experiments

# Record your results here:
# | Epochs | LR    | F1 Score |
# |--------|-------|----------|
# | 3      | 2e-5  |          |
# | 5      | 2e-5  |          |
# | 10     | 2e-5  |          |
# | 5      | 1e-5  |          |
# | 5      | 5e-5  |          |
